# Práctica 2: Determinación de Tipos de Estrellas


| Atributo | Descripción |
|---|---|
| `Temperature` | Temperatura promedio de la superficie (K) |
| `L` | Luminosidad relativa al Sol |
| `R` | Radio relativo al Sol |
| `A_M` | Magnitud absoluta (brillo a 10 parsec) |
| `Spectral_Class` | Clase espectral (O, B, A, F, G, K, M) — de más caliente a más fría |
| `Color` | Color principal del espectro |

In [ ]:
#Instalación de dependencias
%pip install numpy pandas matplotlib seaborn scikit-learn scipy

### 1 - Codificación

A continuación, se van a preparar las variables categóricas (`Color` y `Spectral_Class`) para que los algoritmos de Aprendizaje Automático usados en próximas etapas puedan interpretarlas y calcular distancias correctamente. 

Ambas variables están relacionadas con la cantidad de energía de la estrella, asique utilizaremos una codificación ordinal:

* **Menor valor numérico:** Menor energía / temperatura (Estrellas frías, rojizas, Clase M).
* **Mayor valor numérico:** Mayor energía / temperatura (Estrellas calientes, azuladas, Clase O).

Además, se hará una modificación previa en la columna `Color` para unificar categorías que están escritas de forma distinta (mayúsculas, espacios, etc).

In [ ]:
import pandas as pd
from IPython.display import display

#Cargar el dataset
df = pd.read_csv('stars_data.csv')

#Limpieza de la columna 'Color'
#Lo pasamos todo a minúsculas, quitamos guiones y eliminamos espacios al principio o al final
df['Color'] = df['Color'].str.lower().str.replace('-', ' ').str.strip()

#Clase Espectral
clase_espectral = {
    'M': 0, 
    'K': 1, 
    'G': 2, 
    'F': 3, 
    'A': 4, 
    'B': 5, 
    'O': 6
}

#Color
color = {
    'red': 0,
    'orange red': 1,
    'orange': 1,
    'pale yellow orange': 2,
    'yellowish': 2,
    'yellow white': 3,
    'white yellow': 3,
    'yellowish white': 3,
    'white': 4,
    'whitish': 4,
    'blue white': 5,
    'blue': 6
}

#Aplicar el mapeo creando nuevas columnas
df['Spectral_Class_Encoded'] = df['Spectral_Class'].map(clase_espectral)
df['Color_Encoded'] = df['Color'].map(color)

#Generar tablas de comprobación)
tabla_espectral = df[['Spectral_Class', 'Spectral_Class_Encoded']].drop_duplicates().sort_values('Spectral_Class_Encoded').reset_index(drop=True)
display(tabla_espectral)

tabla_color = df[['Color', 'Color_Encoded']].drop_duplicates().sort_values('Color_Encoded').reset_index(drop=True)
display(tabla_color)

### 2 - Preprocesado y reducción dimensional (PCA)

Antes de aplicar PCA se va a realizar un escalado de los datos, ya que PCA maximiza varianza, y si las variables tienen escalas muy distintas, las de mayor magnitud tendrán una relevancia muy superior.

El dataset tiene variables con distribuciones muy sesgadas (`L` y `R` abarcan varios órdenes de magnitud entre enanas e hipergigantes). Para asegurar un escalado óptimo, se va a comparar 2 escaladores:

- **StandardScaler**
- **RobustScaler**

Se elegirá el que produzca una proyección PCA más informativa de cara a la posterior realización de clustering (mayor varianza y mejor separación visual de grupos).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.decomposition import PCA

#Semilla base del grupo
SEED = 100522189

#Seleccionar características para clustering
caracteristicas_clustering = ['Temperature', 'L', 'R', 'A_M', 'Spectral_Class_Encoded', 'Color_Encoded']
X = df[caracteristicas_clustering]

scalers = {
    'StandardScaler': StandardScaler(),
    'RobustScaler': RobustScaler(),
}

#Creamos gráficos
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle('Comparación de Escaladores', fontsize=14, fontweight='bold', y=1.00)
results = {}
var_totales = {}

for col, (name, sc) in enumerate(scalers.items()):
    X_sc = sc.fit_transform(X)
    pca_tmp = PCA(n_components=2, random_state=SEED)
    X_pc = pca_tmp.fit_transform(X_sc)
    results[name] = {'scaler': sc, 'X_scaled': X_sc, 'X_pca': X_pc, 'pca': pca_tmp}

    var = pca_tmp.explained_variance_ratio_
    var_acum = var.cumsum()
    var_totales[name] = var_acum[1]

    #Proyección PCA
    axes[0, col].scatter(X_pc[:, 0], X_pc[:, 1], alpha=0.7, edgecolors='k', linewidths=0.3, s=50)
    axes[0, col].set_title(f'{name}\nProyección PCA', fontweight='bold')
    axes[0, col].set_xlabel(f'PC1 ({var[0]*100:.1f}%)')
    axes[0, col].set_ylabel(f'PC2 ({var[1]*100:.1f}%)')
    axes[0, col].grid(True, linestyle='--', alpha=0.5)

    #Varianza explicada
    axes[1, col].bar(['PC1', 'PC2'], var * 100, color=['steelblue', 'coral'], edgecolor='k', linewidth=2, label='Individual')
    axes[1, col].plot(['PC1', 'PC2'], var_acum * 100, 'ko--', linewidth=2, markersize=8, label='Acumulada')
    for i, v in enumerate(var):
        axes[1, col].text(i, v * 100 + 1.5, f'{v*100:.1f}%', ha='center', fontsize=11, fontweight='bold')
    axes[1, col].set_title(f'{name}\nVarianza Explicada', fontweight='bold')
    axes[1, col].set_ylabel('Varianza (%)')
    axes[1, col].set_ylim(0, 110)
    axes[1, col].legend(loc='upper right')
    axes[1, col].grid(True, linestyle='--', alpha=0.5)

    print(f"{name}: PC1={var[0]*100:.2f}%  PC2={var[1]*100:.2f}%  Total={var_acum[1]*100:.2f}%")

plt.tight_layout()
plt.show()

In [ ]:

#Seleccionar el mejor escalador y guardar los resultados finales
best_name = max(results, key=lambda k: results[k]['pca'].explained_variance_ratio_[1])
print(f"Escalador seleccionado: {best_name}")
var_best = results[best_name]['pca'].explained_variance_ratio_
print(f"  PC1={var_best[0]*100:.2f}%  PC2={var_best[1]*100:.2f}%  Total={var_best.sum()*100:.2f}%")

#Guardar los objetos finales para usar en clustering
scaler = results[best_name]['scaler']
pca    = results[best_name]['pca']
X_pca_df = pd.DataFrame(results[best_name]['X_pca'], columns=['PC1', 'PC2'])
print(f"\nX_pca_df shape: {X_pca_df.shape}")
display(X_pca_df.head())


#### Resultados del PCA

El resultado final se guarda en:
- **scaler:** el escalador elegido (StandardScaler)
- **X_pca_df:** las 240 estrellas del dataset proyectadas a 2D (columnas PC1 y PC2)
- **pca:** el objeto PCA con la información de varianza

**Escalador seleccionado: StandardScaler**

Aunque RobustScaler obtiene una varianza acumulada total mayor (PC1 + PC2 = 99.6%), concentra casi toda esa varianza en PC1 (98.7%), dejando PC2 con apenas un 0.9%. Esto colapsa la proyección a una dimensión, inutilizando prácticamente PC2 para el clustering.

En cambio, StandardScaler distribuye la varianza de forma más equilibrada (PC1 = 55.8%, PC2 = 29.6%, total 85.4%), generando una proyección 2D donde los puntos están dispersos en ambas dimensiones y los grupos son visualmente distinguibles. Esto es lo que interesa de cara a los algoritmos de clustering.


### 3 - Clustering

Una vez reducidos los datos a 2 dimensiones mediante PCA, se van a aplicar los siguientes tres métodos de clustering:

- **K-Means:** Particionamiento basado en centroides, requiere especificar el número de clústeres **k** antes de entrenar.
- **Clustering Jerárquico:** Se construye una jerarquía de agrupaciones sin necesidad de utilizar **k**. Posteriormente el número de clústeres se elige cortando el dendrograma.
- **DBSCAN:** Se utiliza un algoritmo basado en densidad que detecta clústeres de forma arbitraria y además es capaz de identificar ruido.

Los tres algoritmos se aplicarán sobre **X_pca_df** (PC1 y PC2), comparando sus resultados con las métricas adecuadas para cada uno.

#### 3.1 - K-Means

K-Means divide el espacio en *k* clústeres asignando cada punto al centroide más cercano e iterando hasta convergencia. Es sensible a la escala que ya ha sido cubierta en PCA y requiere definir *k* antes de entrenar.

Se van a utilizar dos métricas complementarias para determinar el número óptimo de clusters:

- **Inercia (método del codo):** Suma de distancias cuadráticas de cada punto a su centroide. Decrece al aumentar *k*; se busca el "codo" donde el descenso se ralentiza.
- **Silhouette score:** Mide la cohesión interna y la separación entre clústeres, toma valores entre −1 y 1. Cuanto más cercano a 1, mejor definidos están los grupos.

Se probará el rango *k* ∈ [2, 15], consideramos 15 un límite superior óptimo ya que se menciona que hay 6 tipos de estrellas lo que nos da un márgen de sobra y así aseguramos capturar el valor óptimo si existe. Se elegirá el valor que maximice el silhouette score y coincida con el codo de la inercia.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

X_km = X_pca_df.values  # array (240, 2)

#Método Elbow (empieza en k=1)
range_elbow = range(1, 16)
inercias = []
for k in range_elbow:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=20)
    km.fit(X_km)
    inercias.append(km.inertia_)

#Silhouette (empieza en k=2)
range_sil = range(2, 16)
silhouettes = []
for k in range_sil:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=20)
    labels = km.fit_predict(X_km)
    silhouettes.append(silhouette_score(X_km, labels))

#Gráficos
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('K-Means: Selección del número de clústeres', fontsize=13, fontweight='bold')

#Codo
axes[0].plot(list(range_elbow), inercias, 'bo-', linewidth=2, markersize=7)
axes[0].set_xlabel('Número de clústeres (k)')
axes[0].set_ylabel('Inercia')
axes[0].set_title('Método Elbow')
axes[0].grid(True, linestyle='--', alpha=0.5)

#Silhouette
axes[1].plot(list(range_sil), silhouettes, 'rs-', linewidth=2, markersize=7)
axes[1].set_xlabel('Número de clústeres (k)')
axes[1].set_ylabel('Silhouette score')
axes[1].set_title('Silhouette Score')
axes[1].grid(True, linestyle='--', alpha=0.5)
best_k = range_sil.start + silhouettes.index(max(silhouettes))
axes[1].axvline(best_k, color='green', linestyle='--', linewidth=1.5, label=f'Mejor k={best_k}')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\nMejor k según silhouette: {best_k}  (score={max(silhouettes):.4f})")
for k, s in zip(range_sil, silhouettes):
    print(f"  k={k}  silhouette={s:.4f}")

In [ ]:
#Entrenar el modelo final con el k óptimo
km_final = KMeans(n_clusters=best_k, random_state=SEED, n_init=20)
km_labels = km_final.fit_predict(X_km)
km_silhouette = silhouette_score(X_km, km_labels)

#Añadir etiquetas al DataFrame
X_pca_df['KMeans_Label'] = km_labels

#Visualización de los clústeres
fig, ax = plt.subplots(figsize=(8, 6))
colors = plt.cm.tab10.colors

for c in range(best_k):
    mask = km_labels == c
    ax.scatter(X_km[mask, 0], X_km[mask, 1],
               color=colors[c], label=f'Clúster {c}',
               alpha=0.8, edgecolors='k', linewidths=0.3, s=60)

#Centroides
centroides = km_final.cluster_centers_
ax.scatter(centroides[:, 0], centroides[:, 1],
           marker='X', s=220, c='black', zorder=5, label='Centroides')

ax.set_title(f'K-Means  (k={best_k},  silhouette={km_silhouette:.4f})', fontsize=12, fontweight='bold')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print(f"Silhouette score final (k={best_k}): {km_silhouette:.4f}")
print(f"Inercia final: {km_final.inertia_:.2f}")
print(f"\nTamaño de cada clúster:")
for c in range(best_k):
    print(f"  Clúster {c}: {(km_labels == c).sum()} estrellas")

#### Resultados K-Means

- **k óptimo: 6:** el silhouette score alcanza su máximo en k=6 (score=0.6525), y el método elbow muestra que a partir de k=5 y k=6 la reducción de inercia se vuelve muy pequeña.
- **Silhouette score: 0.6525:** valor alto (>0.6), esto indica que los clústeres estan bien cohesionados e internamente separados.
- **6 clústeres:** podemos ver que los 6 clusters coinciden con los 6 tipos de estrellas que describe la astronomía, lo cual es una validación indirecta muy sólida del resultado.

Visualmente, los 6 grupos aparecen claramente diferenciados en el espacio PCA 2D, con los centroides bien posicionados en el interior de cada grupo. K-Means produce agrupaciones compactas y esféricas, lo que encaja bien con la estructura de este dataset tras la reducción dimensional.

#### 3.2 - Clustering Jerárquico (Dendrogramas)

En este método de clustering jerárquico se fusionan en cada paso los dos clusters más cercanos, hasta que todos los puntos pertenecen a uno solo. El resultado se visualiza como un **dendrograma**, el corte a una determinada altura determina el número final de clústeres.

A diferencia de K-Means, **no es necesario especificar k de antemano**. Sin embargo, sí hay que elegir el tipo de **linkage**, que define cómo se mide la distancia entre dos clústeres:

- **Ward:** Minimiza el incremento de varianza dentro del cluster al fusionar. Es el más similar a K-Means y suele dar los grupos más compactos.
- **Complete:** Distancia entre los puntos más alejados de cada clúster. Tiende a clústeres compactos y mas alargados.
- **average:** Distancia media entre todos los pares de puntos de cada clúster. Equilibrio entre **ward** y **complete**.

Se compararán los tres linkages visualmente mediante sus dendrogramas y se evaluarán con silhouette score para elegir el mejor.

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram

def plot_dendrogram(model, **kwargs):
    """Extrae la información de un AgglomerativeClustering y dibuja su dendrograma."""
    counts = np.zeros(model.children_.shape[0])
    n_samples = len(model.labels_)
    for i, merge in enumerate(model.children_):
        current_count = 0
        for child_idx in merge:
            if child_idx < n_samples:
                current_count += 1
            else:
                current_count += counts[child_idx - n_samples]
        counts[i] = current_count
    linkage_matrix = np.column_stack([model.children_, model.distances_, counts]).astype(float)
    dendrogram(linkage_matrix, **kwargs)

#Entrenar los 3 modelos con árbol completo para poder dibujar el dendrograma
linkages = ['ward', 'complete', 'average']
modelos_hclust = {}

for lnk in linkages:
    modelo = AgglomerativeClustering(
        metric             = 'euclidean',
        linkage            = lnk,
        distance_threshold = 0,
        n_clusters         = None,
        compute_full_tree  = True
    )
    modelo.fit(X_km)
    modelos_hclust[lnk] = modelo

#Dendrogramas comparativos
fig, axs = plt.subplots(3, 1, figsize=(12, 12))
fig.suptitle('Comparación de Linkages — Dendrogramas', fontsize=14, fontweight='bold')

for ax, lnk in zip(axs, linkages):
    plot_dendrogram(modelos_hclust[lnk], color_threshold=0, ax=ax, no_labels=True)
    ax.set_title(f'Linkage: {lnk}', fontweight='bold')
    ax.set_xlabel('Índice de muestra')
    ax.set_ylabel('Distancia')

plt.tight_layout()
plt.show()

In [ ]:
#Silhouette score para cada linkage y cada k, para elegir el mejor combinación
range_hc = range(2, 16)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Clustering Jerárquico: Silhouette por Linkage', fontsize=13, fontweight='bold')

best_linkage = None
best_hc_k    = None
best_hc_sil  = -1

for ax, lnk in zip(axes, linkages):
    sils = []
    for k in range_hc:
        modelo = AgglomerativeClustering(
            metric     = 'euclidean',
            linkage    = lnk,
            n_clusters = k
        )
        lbl = modelo.fit_predict(X_km)
        sils.append(silhouette_score(X_km, lbl))

    best_k_lnk = range_hc.start + sils.index(max(sils))
    ax.plot(list(range_hc), sils, 'o-', linewidth=2, markersize=6)
    ax.axvline(best_k_lnk, color='green', linestyle='--', linewidth=1.5, label=f'Mejor k={best_k_lnk}')
    ax.set_title(f'Linkage: {lnk}', fontweight='bold')
    ax.set_xlabel('Número de clústeres (k)')
    ax.set_ylabel('Silhouette score')
    ax.legend()
    ax.grid(True, linestyle='--', alpha=0.5)

    print(f"  [{lnk}]  mejor k={best_k_lnk}  silhouette={max(sils):.4f}")

    if max(sils) > best_hc_sil:
        best_hc_sil  = max(sils)
        best_hc_k    = best_k_lnk
        best_linkage = lnk

plt.tight_layout()
plt.show()

print(f"\nMejor combinación: linkage={best_linkage}  k={best_hc_k}  silhouette={best_hc_sil:.4f}")

In [ ]:
#Dendrograma final con el mejor linkage y línea de corte
modelo_final_hc = AgglomerativeClustering(
    metric             = 'euclidean',
    linkage            = best_linkage,
    distance_threshold = 0,
    n_clusters         = None,
    compute_full_tree  = True
)
modelo_final_hc.fit(X_km)

#Calcular altura de corte inspeccionando el dendrograma para best_hc_k clústeres
import scipy.cluster.hierarchy as sch
Z = sch.linkage(X_km, method=best_linkage)
#La altura de corte se obtiene como el punto medio entre las dos últimas fusiones que separan best_hc_k grupos
heights = Z[:, 2]
altura_corte = (heights[-(best_hc_k - 1)] + heights[-(best_hc_k)]) / 2

fig, ax = plt.subplots(figsize=(12, 5))
plot_dendrogram(modelo_final_hc, color_threshold=altura_corte, ax=ax, no_labels=True)
ax.axhline(y=altura_corte, color='red', linestyle='--', linewidth=1.5,
           label=f'Corte → {best_hc_k} clústeres')
ax.set_title(f'Dendrograma final  (linkage={best_linkage},  k={best_hc_k})', fontweight='bold')
ax.set_xlabel('Índice de muestra')
ax.set_ylabel('Distancia')
ax.legend()
plt.tight_layout()
plt.show()

#Modelo final con k elegido y visualización en PCA 2D
modelo_hc_final = AgglomerativeClustering(
    metric     = 'euclidean',
    linkage    = best_linkage,
    n_clusters = best_hc_k
)
hc_labels  = modelo_hc_final.fit_predict(X_km)
hc_silhouette = silhouette_score(X_km, hc_labels)

X_pca_df['HC_Label'] = hc_labels

fig, ax = plt.subplots(figsize=(8, 6))
for c in range(best_hc_k):
    mask = hc_labels == c
    ax.scatter(X_km[mask, 0], X_km[mask, 1],
               color=colors[c], label=f'Clúster {c}',
               alpha=0.8, edgecolors='k', linewidths=0.3, s=60)

ax.set_title(f'Clustering Jerárquico  (linkage={best_linkage},  k={best_hc_k},  silhouette={hc_silhouette:.4f})',
             fontsize=11, fontweight='bold')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
#Tabla comparativa de linkages y tamaño de clústeres
range_hc_sil = range(2, 16)
tabla_datos = []
for lnk in linkages:
    sils_tmp = []
    for k in range_hc_sil:
        m = AgglomerativeClustering(metric='euclidean', linkage=lnk, n_clusters=k)
        sils_tmp.append(silhouette_score(X_km, m.fit_predict(X_km)))
    best_k_tmp = range_hc_sil.start + sils_tmp.index(max(sils_tmp))
    tabla_datos.append([lnk, str(best_k_tmp), f"{max(sils_tmp):.4f}"])

col_labels   = ['Linkage', 'Mejor k', 'Silhouette']
best_row_idx = [row[2] for row in tabla_datos].index(max(row[2] for row in tabla_datos))

#Datos de tamaño de clústeres
cluster_sizes = [[f'Clúster {c}', str((hc_labels == c).sum())] for c in range(best_hc_k)]
col_sizes     = ['Clúster', 'Estrellas']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, max(2.2, best_hc_k * 0.42 + 0.8)))
fig.suptitle(f'Clustering Jerárquico — Resumen  (linkage={best_linkage}, k={best_hc_k})',
             fontsize=11, fontweight='bold')

#Tabla comparación linkages
ax1.axis('off')
t1 = ax1.table(cellText=tabla_datos, colLabels=col_labels, cellLoc='center', loc='center')
t1.auto_set_font_size(False)
t1.set_fontsize(10)
t1.scale(1, 1.6)
for j in range(len(col_labels)):
    t1[0, j].set_facecolor('#404040')
    t1[0, j].set_text_props(color='white', fontweight='bold')
for i, _ in enumerate(tabla_datos):
    color_row = '#c6efce' if i == best_row_idx else ('#f2f2f2' if i % 2 == 0 else 'white')
    bold = i == best_row_idx
    for j in range(len(col_labels)):
        t1[i + 1, j].set_facecolor(color_row)
        if bold:
            t1[i + 1, j].set_text_props(fontweight='bold')
ax1.set_title('Comparación de linkages', fontsize=10, fontweight='bold', pad=8)

#Tabla tamaño de clústeres
ax2.axis('off')
t2 = ax2.table(cellText=cluster_sizes, colLabels=col_sizes, cellLoc='center', loc='center')
t2.auto_set_font_size(False)
t2.set_fontsize(10)
t2.scale(1, 1.6)
for j in range(len(col_sizes)):
    t2[0, j].set_facecolor('#404040')
    t2[0, j].set_text_props(color='white', fontweight='bold')
for i in range(len(cluster_sizes)):
    for j in range(len(col_sizes)):
        t2[i + 1, j].set_facecolor('#f2f2f2' if i % 2 == 0 else 'white')
ax2.set_title('Tamaño de cada clúster', fontsize=10, fontweight='bold', pad=8)

plt.tight_layout()
plt.show()

#### Resultados Clustering Jerárquico

Comparación de los tres linkages mediante silhouette score, como muestra la tabla:

- **Linkage seleccionado automáticamente: complete** con k=8 y silhouette=0.6433, el valor más alto entre los tres linkages evaluados. Sin embargo, esta elección tiene una limitación importante: **complete** es especialmente sensible a outliers, y al buscar el k que maximiza el Silhouette, tiende a crear clústeres muy pequeños (1–4 estrellas) para aislar puntos extremos, en lugar de agruparlos dentro del grupo real al que pertenecen.
- **Por qué no se elige ward:** ward con k=6 obtiene un Silhouette ligeramente inferior (≈0.64), pero genera clústeres más equilibrados y coincide con los 6 tipos estelares conocidos. Aun así, dado que el criterio de selección del algoritmo es la métrica (Silhouette), se mantiene **complete** como ganador de esta fase. 

Visualmente, el dendrograma con **complete** y k=8 muestra dos o tres clústeres muy pequeños (1–4 estrellas) junto a los grupos principales. Esto refleja una debilidad de que al medir la distancia entre clústeres como la distancia máxima entre sus puntos, los outliers o estrellas tienden a formar grupos propios en lugar de fusionarse con su grupo más cercano.

#### 3.3 - DBSCAN

DBSCAN es un algoritmo de clustering basado en densidad, este método no requiere especificar el número de clústeres de antemano. Identifica zonas de alta densidad separadas por zonas de baja densidad y es capaz de detectar **ruido**.

Los dos hiperparámetros principales son:

- **eps (ε):** radio del vecindario alrededor de cada punto. Dos puntos son vecinos si su distancia de separación es menor o igual que eps.
- **min_samples:** número mínimo de puntos en el vecindario de eps para que un punto se considere **core point**. Los puntos con menos vecinos son etiquetados como **ruido**.

A diferencia de K-Means y el clustering jerárquico, DBSCAN puede formar clústeres de forma arbitraria y no asigna todos los puntos a un grupo. La métrica de evaluación usada será **DBCV**, que es específica para algoritmos basados en densidad, ya que el silhouette score no penaliza adecuadamente el ruido y asume que los clústeres son convexos. DBCV no está disponible en scikit-learn y requiere una librería externa.

El ajuste de hiperparámetros se realiza en dos pasos:
- **Heurística k-distance:** estimar un eps inicial analizando la distancia al k-ésimo vecino más cercano de cada punto y buscando el codo de la curva.
- **Grid search con DBCV:** probar combinaciones de eps y min_samples alrededor del valor heurístico y seleccionar la que maximice el DBCV score.

In [ ]:
#Instalación de la librería DBCV (métrica específica para DBSCAN)
%pip install "git+https://github.com/FelSiq/DBCV"

In [ ]:
from sklearn.neighbors import NearestNeighbors

min_samples_ref = 5  #Valor de referencia para la heurística

#Calcular distancia al k-ésimo vecino para cada punto
nn = NearestNeighbors(n_neighbors=min_samples_ref)
nn.fit(X_km)
distances, _ = nn.kneighbors(X_km)

#Ordenar distancias al k-ésimo vecino
sorted_distances = np.sort(distances[:, min_samples_ref - 1])

#Detectar el codo automáticamente haciendo la segunda derivada máxima
diffs2 = np.diff(sorted_distances, n=2)
idx_codo = int(np.argmax(diffs2)) + 1
eps_heuristic = float(sorted_distances[idx_codo])

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(sorted_distances, linewidth=2, color='steelblue')
ax.axvline(x=idx_codo, color='red', linestyle='--', linewidth=1.5,
           label=f'Codo (índice={idx_codo})')
ax.axhline(y=eps_heuristic, color='green', linestyle='--', linewidth=1.5,
           label=f'eps estimado ≈ {eps_heuristic:.3f}')
ax.set_xlabel(f'Puntos ordenados por distancia al {min_samples_ref}-ésimo vecino')
ax.set_ylabel('Distancia')
ax.set_title(f'k-distance plot  (min_samples={min_samples_ref})')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print(f"eps heurístico estimado: {eps_heuristic:.4f}")

In [ ]:
import dbcv as dbcv_module
from sklearn.cluster import DBSCAN

#Valores a probar alrededor del eps heurístico
eps_values       = np.round(np.linspace(eps_heuristic * 0.5, eps_heuristic * 3.0, 10), 3)
min_samples_vals = [3, 4, 5, 6, 7]

best_dbcv_score = -np.inf
best_eps_db     = None
best_ms_db      = None

print(f"{'MS':<3} | {'EPS':<7} | {'Clusters':<10} | {'% Ruido':<8} | {'DBCV Score'}")
print("-" * 52)

for ms in min_samples_vals:
    for eps in eps_values:
        db  = DBSCAN(eps=float(eps), min_samples=ms, metric='euclidean').fit(X_km)
        lbl = db.labels_
        n_cls = len(set(lbl) - {-1})
        n_nse = int((lbl == -1).sum())
        pct   = n_nse / len(lbl) * 100

        if n_cls > 1:
            try:
                score = dbcv_module.dbcv(X_km, lbl)
            except Exception:
                score = float('nan')
            marker = ' ← mejor' if score > best_dbcv_score else ''
            if score > best_dbcv_score:
                best_dbcv_score = score
                best_eps_db     = float(eps)
                best_ms_db      = ms
            print(f"{ms:<3} | {eps:<7.3f} | {n_cls:<10} | {pct:>7.1f}% | {score:.4f}{marker}")
        else:
            print(f"{ms:<3} | {eps:<7.3f} | {n_cls:<10} | {pct:>7.1f}% | No calculable")

print(f"\nMejor combinación: eps={best_eps_db:.3f}  min_samples={best_ms_db}  DBCV={best_dbcv_score:.4f}")

In [ ]:
#Modelo DBSCAN final con los mejores hiperparámetros
db_final  = DBSCAN(eps=best_eps_db, min_samples=best_ms_db, metric='euclidean')
db_labels = db_final.fit_predict(X_km)

n_clusters_db = len(set(db_labels) - {-1})
n_noise_db    = int((db_labels == -1).sum())
db_dbcv_score = dbcv_module.dbcv(X_km, db_labels)

X_pca_df['DBSCAN_Label'] = db_labels

#Visualización en el espacio PCA 2D
fig, ax = plt.subplots(figsize=(8, 6))
unique_cls = sorted(c for c in set(db_labels) if c != -1)

for i, c in enumerate(unique_cls):
    mask = db_labels == c
    ax.scatter(X_km[mask, 0], X_km[mask, 1],
               color=colors[i % len(colors)], label=f'Clúster {c}',
               alpha=0.8, edgecolors='k', linewidths=0.3, s=60)

#Ruido marcados con X negra
if n_noise_db > 0:
    mask_noise = db_labels == -1
    ax.scatter(X_km[mask_noise, 0], X_km[mask_noise, 1],
               color='black', marker='x', s=80, linewidths=1.5,
               label=f'Ruido ({n_noise_db})', zorder=5)

ax.set_title(f'DBSCAN  (eps={best_eps_db:.3f},  min_samples={best_ms_db},  DBCV={db_dbcv_score:.4f})',
             fontsize=11, fontweight='bold')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print(f"Clústeres encontrados: {n_clusters_db}")
print(f"Outliers (ruido): {n_noise_db}  ({n_noise_db / len(db_labels) * 100:.1f}%)")
print(f"DBCV score: {db_dbcv_score:.4f}")
print(f"\nTamaño de cada clúster:")
for c in sorted(set(db_labels)):
    label_str = "Ruido (-1)" if c == -1 else f"Clúster {c}"
    print(f"  {label_str}: {(db_labels == c).sum()} estrellas")

#### Resultados DBSCAN

- **eps óptimo: 0.481:** se estimó un eps heurístico de ≈ 0.962, pero el grid search con DBCV muestra que encontró valores más pequeños (0.481) los cuales producen clústeres más densos y mejor puntuados.
- **min_samples óptimo:** es valor más óptimo encontrado fue el **5**.
- **DBCV score: 0.8214:** es el valor más alto de los tres algoritmos. Nos basamos en los resultados de DBCV ya que es la métrica más adecuada para clustering basado en densidad, porque penaliza la dispersión interna y la proximidad entre clústeres de forma específica para este tipo de agrupaciones.
- **Clústeres encontrados: 4:** DBSCAN agrupa las estrellas en 4 grupos densos, detectando estructuras compactas y bien diferenciadas en el espacio PCA.
- **Outliers (ruido): 28 estrellas (11.7%):** DBSCAN considera que son puntos aislados que no pertenecen a ningún clúster denso. Corresponden principalmente a estrellas dispersas entre los grupos compactos.

La limitación principal de DBSCAN es que solo encuentra **4 clústeres** cuando hay 6 tipos de estrellas conocidos. Los grupos densos identificados corresponden a las familias más compactas del dataset: las **enanas marrones** (grupo frío y compacto de baja luminosidad), las **supergigantes rojas** (temperatura baja pero luminosidad muy alta), las **hipergigantes rojas** (las de mayor radio del dataset) y un grupo mixto de **supergigantes e hipergigantes calientes**. Por otro lado, las estrellas de **secuencia principal** y parte de las supergigantes calientes quedan clasificadas como ruido, ya que en el espacio PCA forman una zona de densidad más continua que no alcanza el umbral mínimo de densidad definido por los hiperparámetros seleccionados.

### 4 - Pipeline de clustering recomendado

Una vez evaluados los tres algoritmos, se van a comparan sus resultados para poder seleccionar el pipeline más adecuado para este problema.

La comparación se realiza teniendo en cuenta:

- **Métrica de calidad:** Silhouette (K-Means y HC) y DBCV (DBSCAN). Se calcula el Silhouette de DBSCAN (excluyendo los puntos de ruido) para poder comparar los tres métodos de la misma forma, ya que DBCV es específico para clustering basado en densidad y no es directamente comparable con Silhouette.
- **Número de clústeres encontrado:** se valora que coincida con los 6 tipos estelares conocidos.
- **Cobertura:** porcentaje de estrellas asignadas a un clúster (sin ruido).
- **Interpretabilidad:** facilidad para asociar cada clúster a un tipo estelar.

In [ ]:

from sklearn.metrics import silhouette_score

#Silhouette de DBSCAN excluyendo puntos de ruido
mask_valid = db_labels != -1
sil_dbscan_valid = silhouette_score(X_km[mask_valid], db_labels[mask_valid]) if mask_valid.sum() > 1 else float('nan')

#Tabla comparativa de los tres algoritmos
tabla_comp = pd.DataFrame({
    'Algoritmo':        ['K-Means',           'Jerárquico (HC)',                              'DBSCAN'],
    'Hiperparámetros':  [f'k = {best_k}',     f'linkage = {best_linkage}, k = {best_hc_k}',  f'eps = {best_eps_db:.3f}, min_samples = {best_ms_db}'],
    'Nº Clústeres':     [best_k,              best_hc_k,                                     n_clusters_db],
    'Ruido (pts)':      ['0 (0%)',            '0 (0%)',                                       f'{n_noise_db} ({n_noise_db/len(db_labels)*100:.1f}%)'],
    'Silhouette':       [f'{km_silhouette:.4f}',  f'{best_hc_sil:.4f}',                      f'{sil_dbscan_valid:.4f} *'],
    'DBCV':             ['—',                 '—',                                            f'{db_dbcv_score:.4f}'],
})

display(tabla_comp.set_index('Algoritmo'))
print(f"\nSilhouette de DBSCAN calculado solo sobre los {int(mask_valid.sum())} puntos no etiquetados como ruido.")


#### Pipeline recomendado: K-Means con k = 6

A partir de los resultados obtenidos, el pipeline recomendado es el siguiente:

StandardScaler **PCA** (n_components=2, random_state=SEED) con **KMeans** (n_clusters=6, n_init=20, random_state=SEED).

**Justificación:**

- **Número de clústeres:** K-Means con k=6 es el único algoritmo que recupera exactamente los 6 tipos estelares que se describen. El clustering jerárquico con **complete** necesita k=8 para maximizar su Silhouette, generando dos clústeres de pocas estrellas que dividen innecesariamente un grupo real. Por otro lado DBSCAN solo detecta 4 grupos y deja un 11.7% de estrellas como ruido.

- **Calidad de los clústeres:** El Silhouette de K-Means (0.6525) es el más alto entre los métodos de particionado, con clústeres compactos y bien separados en el espacio PCA 2D.

- **Cobertura total:** A diferencia de DBSCAN, K-Means asigna todas las estrellas a un grupo, lo que es correcto para este problema ya que el enunciado indica que hay 240 estrellas y 6 grupos bien definidos, por lo que no hay razón para descartar ninguna estrella como ruido.

- **Reproducibilidad y simplicidad:** Fijando **random_state=SEED**, los resultados son completamente reproducibles. El modelo es sencillo de interpretar y no requiere estimaciones de densidad ni umbrales de distancia adicionales.

- **Adecuación a la estructura de los datos:** Tras la reducción PCA con StandardScaler (85.4% de varianza explicada), los grupos forman regiones compactas y con forma esférica, que es la geometría asumida por K-Means y para la que el algoritmo está optimizado.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

#Pipeline recomendado: StandardScaler → PCA(2) → KMeans(k=6)
pipeline_scaler = StandardScaler()
pipeline_pca    = PCA(n_components=2, random_state=SEED)
pipeline_kmeans = KMeans(n_clusters=6, n_init=20, random_state=SEED)

X_pipeline = df[['Temperature', 'L', 'R', 'A_M', 'Spectral_Class_Encoded', 'Color_Encoded']]
X_scaled   = pipeline_scaler.fit_transform(X_pipeline)
X_reduced  = pipeline_pca.fit_transform(X_scaled)
pipeline_labels = pipeline_kmeans.fit_predict(X_reduced)

print("Pipeline recomendado:")
print(f"  StandardScaler: PCA(n_components=2, random_state={SEED}) con KMeans(n_clusters=6, n_init=20, random_state={SEED})")
print(f"\nVarianza explicada: PC1={pipeline_pca.explained_variance_ratio_[0]*100:.1f}%  PC2={pipeline_pca.explained_variance_ratio_[1]*100:.1f}%  Total={pipeline_pca.explained_variance_ratio_.sum()*100:.1f}%")
print(f"Silhouette score:  {silhouette_score(X_reduced, pipeline_labels):.4f}")
print(f"Inercia:           {pipeline_kmeans.inertia_:.2f}")
print(f"\nDistribución de clústeres:")
for c in range(6):
    print(f"  Clúster {c}: {(pipeline_labels == c).sum()} estrellas")


### 5 - Comparación con las clases astronómicas

El objetivo de este apartado es contrastar los grupos obtenidos por el pipeline recomendado (K-Means, k=6) con las seis clases estelares que la astronomía utiliza habitualmente:

| Clase | Temperatura (K) | L | R | A_M | Color | Clase Espectral |
|---|---|---|---|---|---|---|
| Enana roja | 3.000 | 7,0·10⁻⁴ | 1,0·10⁻¹ | +17,5 | rojo | K-M |
| Enana marrón | 3.300 | 5,5·10⁻³ | 3,5·10⁻¹ | +12,5 | rojo | M |
| Enana blanca | 14.000 | 2,5·10⁻³ | 1,0·10⁻² | +12,6 | blanca | B-G |
| Sec. principal | 16.000 | 3,2·10⁴ | 4,4 | −0,4 | blanca-amarilla | B-M |
| Supergigante | 15.000 | 3,0·10⁵ | 5,0·10¹ | −6,4 | blanca-amarilla | B-M |
| Hipergigante | 11.000 | 3,0·10⁵ | 1,4·10³ | −9,6 | amarilla | B-M |

Para realizar la comparación se calcula el perfil medio (valores originales, sin escalar) de cada clúster obtenido por K-Means y se intenta asociarlo a la clase astronómica más similar.

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from IPython.display import display

#Perfil medio de cada clúster (espacio original, sin escalar)
df_perfil = df.copy()
df_perfil['Cluster'] = km_labels

cols_num = ['Temperature', 'L', 'R', 'A_M']
perfil = df_perfil.groupby('Cluster')[cols_num].mean().round(4)
perfil['Tamaño'] = df_perfil.groupby('Cluster').size()
perfil['Color_freq']    = df_perfil.groupby('Cluster')['Color'].agg(lambda x: x.value_counts().index[0])
perfil['Spectral_freq'] = df_perfil.groupby('Cluster')['Spectral_Class'].agg(lambda x: x.value_counts().index[0])

#Ordenar clústeres de menor a mayor temperatura media para facilitar la comparación
perfil_sorted = perfil.sort_values('Temperature').reset_index()
print("Perfil medio por clúster K-Means (ordenado por Temperatura): ")
display(perfil_sorted.rename(columns={
    'Cluster': 'Clúster', 'Temperature': 'Temp (K)',
    'L': 'L (rel. Sol)', 'R': 'R (rel. Sol)', 'A_M': 'A_M',
    'Color_freq': 'Color más frecuente', 'Spectral_freq': 'Clase Espec. más frecuente'
}))

#Referencia astronómica ajustada a la distribución del dataset
referencia = pd.DataFrame({
    'Tipo Estelar':    ['Enana marrón', 'Supergigante roja', 'Hipergigante roja', 'Sec. principal', 'Supergigante', 'Hipergigante'],
    'Temp (K)':        [3200,    3500,    3800,    13000,  22000,   24000],
    'L (rel. Sol)':    [0.0175,  206600,  245541,  134,    203221,  512365],
    'R (rel. Sol)':    [0.26,    129,     1392,    1.25,   53,      1105],
    'A_M':             [14.6,    -6.8,    -9.9,    8.4,    -5.6,    -8.7],
}).set_index('Tipo Estelar')

print("\nReferencia del dataset (perfil medio por clúster): ")
display(referencia)

#Asignación directa basada en la inspección del perfil real
tipo_por_cluster = {
    3: 'Enana marrón',
    0: 'Supergigante roja',
    4: 'Hipergigante roja',
    2: 'Sec. principal',
    1: 'Supergigante',
    5: 'Hipergigante',
}
perfil_sorted['Tipo asignado'] = perfil_sorted['Cluster'].map(tipo_por_cluster)

print("\nCorrespondencia Clúster → Tipo Estelar: ")
display(perfil_sorted[['Cluster', 'Temperature', 'L', 'R', 'A_M', 'Tipo asignado']].rename(
    columns={'Cluster': 'Clúster', 'Temperature': 'Temp (K)', 'L': 'L (rel. Sol)', 'R': 'R (rel. Sol)'}
))


In [ ]:

#Perfil de cada clúster vs referencia del dataset (barras agrupadas por pares)
ref_vals = {
    'Enana marrón':       [3200,  0.0175,  0.26,   14.6],
    'Supergigante roja':  [3500,  206600,  129,    -6.8],
    'Hipergigante roja':  [3800,  245541,  1392,   -9.9],
    'Sec. principal':     [13000, 134,     1.25,    8.4],
    'Supergigante':       [22000, 203221,  53,     -5.6],
    'Hipergigante':       [24000, 512365,  1105,   -8.7],
}

variables  = ['Temperature', 'L', 'R', 'A_M']
etiquetas  = ['Temperatura (K)', 'Luminosidad (rel. Sol)', 'Radio (rel. Sol)', 'Magnitud absoluta (A_M)']
tipos_asignados = perfil_sorted['Tipo asignado'].tolist()

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Perfil medio de los clústeres K-Means vs. referencia del dataset', fontsize=13, fontweight='bold')

cluster_colors_list = [plt.cm.tab10(i) for i in range(len(perfil_sorted))]
n = len(perfil_sorted)
bar_width = 0.38
x_centers = np.arange(n) * 2.2

for idx, (ax, var, etiq) in enumerate(zip(axes.ravel(), variables, etiquetas)):
    cluster_vals  = perfil_sorted[var].values
    ref_vals_list = [ref_vals[tipo][idx] for tipo in tipos_asignados]

    ax.bar(x_centers - bar_width / 2, cluster_vals, width=bar_width,
           color=cluster_colors_list, edgecolor='k', linewidth=0.8, label='Clúster K-Means')
    ax.bar(x_centers + bar_width / 2, ref_vals_list, width=bar_width,
           color=cluster_colors_list, edgecolor='k', linewidth=0.8,
           hatch='//', alpha=0.7, label='Referencia del dataset')

    ax.set_xticks(x_centers)
    ax.set_xticklabels(
        [f'C{int(r["Cluster"])}\n({tipos_asignados[i]})' for i, (_, r) in enumerate(perfil_sorted.iterrows())],
        rotation=30, ha='right', fontsize=7
    )
    ax.set_title(etiq, fontweight='bold')
    ax.set_yscale('symlog' if var in ['L', 'R'] else 'linear')
    ax.grid(True, linestyle='--', alpha=0.4, axis='y')
    if idx == 0:
        ax.legend(fontsize=8)

plt.tight_layout()
plt.show()



#### Discusión: similitudes entre los clústeres y las clases astronómicas

La comparación entre los seis clústeres obtenidos con K-Means y los tipos estelares permite concluir que la correspondencia es muy alta. Aun así, la interpretación no coincide por completo con las clases astronómicas canónicas, ya que este dataset está dominado por **supergigantes e hipergigantes rojas frías**, en lugar de incluir una representación más equilibrada de otros tipos como las enanas blancas.

**Correspondencia entre clústeres y tipos estelares:**

- **C3: Enana marrón** (T≈3.213 K, L≈0.017, R≈0.26, A_M≈+14.6). Es el clúster más numeroso, con 84 estrellas. Presenta una temperatura muy baja, una luminosidad muy reducida y un radio subsolar, rasgos propios de objetos fríos y poco brillantes. La magnitud absoluta tan positiva refuerza esta interpretación, por lo que su asociación con una **enana marrón** resulta coherente.

- **C0: Supergigante roja** (T≈3.467 K, L≈206.600, R≈130, A_M≈−6.8). Agrupa solo 10 estrellas, pero con un perfil muy característico: temperatura baja y, al mismo tiempo, valores muy altos de luminosidad y radio. Esta combinación es propia de las **supergigantes rojas frías**.

- **C4: Hipergigante roja** (T≈3.782 K, L≈245.541, R≈1.392, A_M≈−9.9). Son 24 estrellas y representa el grupo con radios más extremos del dataset. Su tamaño enorme y su elevada luminosidad encajan con el perfil de una **hipergigante roja**, una de las clases más masivas y luminosas entre las estrellas frías.

- **C2: Secuencia principal** (T≈12.522 K, L≈134, R≈1.25, A_M≈+8.4). Incluye 62 estrellas con temperatura moderadamente alta, luminosidad contenida y radio pequeño. La magnitud absoluta positiva indica que no se trata de estrellas muy brillantes, lo que la sitúa de forma razonable dentro de la **secuencia principal**.

- **C1: Supergigante** (T≈21.793 K, L≈203.222, R≈53, A_M≈−5.6). Este grupo contiene 42 estrellas con temperatura alta, gran luminosidad y radio elevado. Su perfil es consistente con el de **supergigantes calientes**, probablemente de tipo azul o blanco.

- **C5: Hipergigante** (T≈24.016 K, L≈512.366, R≈1.105, A_M≈−8.7). Formado por 18 estrellas, reúne los valores más extremos de temperatura, luminosidad y tamaño dentro de los grupos calientes. Por ello, su interpretación como **hipergigante** es la más adecuada.

**Diferencias respecto a la referencia canónica:**

Las tablas astronómicas del enunciado suelen describir supergigantes e hipergigantes con temperaturas intermedias, en torno a 11.000-15.000 K. Sin embargo, en este dataset aparecen también **supergigantes e hipergigantes rojas frías** con temperaturas inferiores a 4.000 K, que no suelen figurar en las referencias más resumidas. Además, no se puede identificar un grupo de **enanas blancas**, ni tampoco una población de **enanas rojas** con luminosidades extremadamente bajas. Por el contrario, aparece un grupo mejor descrito como enanas marrones frías. Por tanto, las diferencias con la referencia no indican un fallo del clustering, sino una particularidad de la distribución real de los datos.

**Conclusión:**

En conjunto, K-Means identifica con gran precisión los seis grandes grupos que aparecen en el dataset. La correspondencia con categorías estelares físicas es muy sólida, gracias a la distinción de **L** y **R**, que separan objetos compactos, supergigantes e hipergigantes. La variable **A_M** también ayuda de forma importante al distinguir estrellas muy brillantes de objetos débiles, mientras que la **temperatura** actúa principalmente para separar grupos fríos y calientes. Con todo esto podemos decir que la partición obtenida es consistente y fácilmente interpretable.


### Uso de IA
Se ha utilizado la IA para ayudarnos en las siguientes tareas:
- Corregir algunos errores de sintaxis
- Depuración de codigo
- Asistencia en la generacion de graficos.
